before starting genie code - select desired compute in top right, I use serverless v5 if allowed, otherwise use your own personal all purpose compute cluster

## the objective of this notebook is to read data from zip files stored in a volume into appropriate delta tables to do downstream analysis on the Seattle market

Ingest Steps:
- read data in from zip files in raw_data schema and create appropriate delta tables
- Use cmegdemos_catalog.network_enablement_demo (parameterize these values throughout the notebook)
- raw data zip files are in the raw_data volume in this schema
- Load FCC BDC H3 hexagon coverage data - filter to only Seattle area
- Importing Microsoft building footprints as polygon geometries using Databricks Spatial SQL and save to delta table 
- load in 310.csv.gz file from Open Cell ID for tower data in seatle market but filter to only t mobile
- make sure tables are ready for downstream analysis with H3 and ST functions

prompt to genie code: see instructions in this notebook and get to work, feel free to add an additional markdown below with refined instructions after you create your plan and/or ask any questions to me before starting on the work

## Refined Ingestion Plan

**Target**: `cmegdemos_catalog.network_enablement_demo`  
**Source**: `/Volumes/cmegdemos_catalog/network_enablement_demo/raw_data/`  
**Seattle Bounding Box**: Lat 47.3–47.8, Lon -122.5 to -122.0

### Files Discovered
| File | Format | Description |
| --- | --- | --- |
| `bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip` | GeoPackage (.gpkg) | FCC BDC 5G NR coverage at H3 res-9. Cols: technology, mindown, minup, environmnt, h3_res9_id, geometry |
| `Washington.zip` | Shapefile (.shp) | Microsoft building footprints for WA state. Cols: Height, geometry (POLYGON, EPSG:4326) |
| `310.csv.gz` | CSV (no header) | OpenCellID towers for MCC 310 (US). ~510K rows. Standard cols: radio, mcc, net, area, cell, unit, lon, lat, range, samples, etc. |

### Ingestion Steps
1. **Setup** — parameterize catalog, schema, volume path
2. **FCC BDC H3 coverage** — extract gpkg → filter to Seattle bbox → save Delta table with `h3_res9_id` (STRING) for H3 functions
3. **Microsoft building footprints** — extract shapefile → filter to Seattle bbox → save Delta table with `geometry GEOMETRY(4326)` for ST functions
4. **OpenCellID towers** — read CSV with proper headers → filter T-Mobile (MNC 260) + Seattle bbox → save Delta table with point geometry
5. **Validation** — confirm all tables are populated and spatial-query ready

In [0]:
%pip install geopandas fiona pyogrio -q

In [0]:
%restart_python

In [0]:
# ── Parameterized configuration ──────────────────────────────────────────────
CATALOG = "cmegdemos_catalog"
SCHEMA = "network_enablement_demo"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

# Seattle metro bounding box
SEATTLE_LAT_MIN, SEATTLE_LAT_MAX = 47.3, 47.8
SEATTLE_LON_MIN, SEATTLE_LON_MAX = -122.5, -122.0

# Target table names
FCC_TABLE = f"{CATALOG}.{SCHEMA}.fcc_bdc_5g_h3_seattle"
BUILDINGS_TABLE = f"{CATALOG}.{SCHEMA}.ms_building_footprints_seattle"
TOWERS_TABLE = f"{CATALOG}.{SCHEMA}.opencellid_tmobile_seattle"

print(f"Catalog:  {CATALOG}")
print(f"Schema:   {SCHEMA}")
print(f"Volume:   {VOLUME_PATH}")
print(f"Seattle bbox: lat [{SEATTLE_LAT_MIN}, {SEATTLE_LAT_MAX}], lon [{SEATTLE_LON_MIN}, {SEATTLE_LON_MAX}]")
print(f"\nTarget tables:")
print(f"  FCC BDC 5G H3:    {FCC_TABLE}")
print(f"  Building foots:   {BUILDINGS_TABLE}")
print(f"  Tower data:       {TOWERS_TABLE}")

### Step 1 — FCC BDC 5G NR H3 Coverage (Seattle)
Extract GeoPackage from zip → filter to Seattle bounding box → save as Delta table with `h3_res9_id` for downstream H3 analysis.

In [0]:
import zipfile
import geopandas as gpd
import pandas as pd

# Extract gpkg from zip
extract_dir = "/tmp/fcc_bdc"
zf_path = f"{VOLUME_PATH}/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.zip"

with zipfile.ZipFile(zf_path, 'r') as zf:
    zf.extractall(extract_dir)

gpkg_path = f"{extract_dir}/bdc_53_5GNR_mobile_broadband_h3_J25_14apr2026.gpkg"

# Read full geopackage and filter to Seattle bounding box
print("Reading FCC BDC GeoPackage...")
gdf_fcc = gpd.read_file(
    gpkg_path,
    bbox=(SEATTLE_LON_MIN, SEATTLE_LAT_MIN, SEATTLE_LON_MAX, SEATTLE_LAT_MAX)
)
print(f"Rows after Seattle bbox filter: {len(gdf_fcc):,}")

# Convert to Spark DataFrame - keep h3_res9_id as string, drop gpd geometry
# (H3 hex boundaries can be regenerated from h3_res9_id using H3 functions)
pdf_fcc = gdf_fcc.drop(columns=["geometry"]).copy()
pdf_fcc["technology"] = pdf_fcc["technology"].astype(int)
pdf_fcc["environmnt"] = pdf_fcc["environmnt"].astype(int)

df_fcc = spark.createDataFrame(pdf_fcc)

# Write to Delta table
df_fcc.write.mode("overwrite").saveAsTable(FCC_TABLE)
print(f"\n✅ Saved {FCC_TABLE}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {FCC_TABLE}").show()
spark.sql(f"SELECT * FROM {FCC_TABLE} LIMIT 5").show(truncate=False)

### Step 2 — Microsoft Building Footprints (Seattle)
Extract shapefile from zip → filter to Seattle bounding box → save as Delta table with `GEOMETRY(4326)` polygon column for ST function analysis.

In [0]:
import zipfile
import geopandas as gpd

# Extract shapefile from zip
extract_dir = "/tmp/wa_buildings"
zf_path = f"{VOLUME_PATH}/Washington.zip"

with zipfile.ZipFile(zf_path, 'r') as zf:
    zf.extractall(extract_dir)

# Read shapefile filtered to Seattle bounding box
print("Reading Washington building footprints shapefile...")
gdf_bldg = gpd.read_file(
    f"{extract_dir}/bldg_footprints.shp",
    bbox=(SEATTLE_LON_MIN, SEATTLE_LAT_MIN, SEATTLE_LON_MAX, SEATTLE_LAT_MAX)
)
print(f"Buildings in Seattle bbox: {len(gdf_bldg):,}")

# Convert geometry to WKT string for Spark ingestion
gdf_bldg["geometry_wkt"] = gdf_bldg["geometry"].to_wkt()
pdf_bldg = gdf_bldg[["Height", "geometry_wkt"]].copy()
pdf_bldg.columns = ["height", "geometry_wkt"]

# Create Spark DataFrame with WKT strings
df_bldg = spark.createDataFrame(pdf_bldg)
df_bldg.createOrReplaceTempView("bldg_staging")

# Create Delta table with native GEOMETRY(4326) column using Spatial SQL
spark.sql(f"DROP TABLE IF EXISTS {BUILDINGS_TABLE}")
spark.sql(f"""
    CREATE TABLE {BUILDINGS_TABLE} AS
    SELECT
        height,
        ST_GeomFromWKT(geometry_wkt, 4326) AS geometry
    FROM bldg_staging
""")

print(f"\n✅ Saved {BUILDINGS_TABLE}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {BUILDINGS_TABLE}").show()
spark.sql(f"SELECT height, ST_AsText(geometry) AS geom_wkt FROM {BUILDINGS_TABLE} LIMIT 5").show(truncate=False)

### Step 3 — OpenCellID Tower Data (T-Mobile, Seattle)
Read `310.csv.gz` (headerless CSV, MCC 310 = US) → apply standard OpenCellID column names → filter T-Mobile (MNC 260) + Seattle bounding box → save as Delta table with `GEOMETRY(4326)` point column.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType

# OpenCellID standard schema (file has no header row)
schema = StructType([
    StructField("radio", StringType()),
    StructField("mcc", IntegerType()),
    StructField("net", IntegerType()),
    StructField("area", IntegerType()),
    StructField("cell", LongType()),
    StructField("unit", IntegerType()),
    StructField("lon", DoubleType()),
    StructField("lat", DoubleType()),
    StructField("range", IntegerType()),
    StructField("samples", IntegerType()),
    StructField("changeable", IntegerType()),
    StructField("created", LongType()),
    StructField("updated", LongType()),
    StructField("averageSignal", IntegerType()),
])

df_towers = (
    spark.read
    .option("header", "false")
    .schema(schema)
    .csv(f"{VOLUME_PATH}/310.csv.gz")
)

# Filter to T-Mobile (MNC 260) and Seattle bounding box
df_towers_seattle = df_towers.filter(
    (df_towers.net == 260)
    & (df_towers.lat >= SEATTLE_LAT_MIN) & (df_towers.lat <= SEATTLE_LAT_MAX)
    & (df_towers.lon >= SEATTLE_LON_MIN) & (df_towers.lon <= SEATTLE_LON_MAX)
)

print(f"T-Mobile towers in Seattle bbox: {df_towers_seattle.count()}")

# Stage and create table with native GEOMETRY point column
df_towers_seattle.createOrReplaceTempView("towers_staging")

spark.sql(f"DROP TABLE IF EXISTS {TOWERS_TABLE}")
spark.sql(f"""
    CREATE TABLE {TOWERS_TABLE} AS
    SELECT
        radio,
        mcc,
        net,
        area,
        cell,
        unit,
        lon,
        lat,
        `range`,
        samples,
        changeable,
        created,
        updated,
        averageSignal,
        ST_Point(lon, lat, 4326) AS geometry
    FROM towers_staging
""")

print(f"\n✅ Saved {TOWERS_TABLE}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {TOWERS_TABLE}").show()
spark.sql(f"SELECT radio, net, lat, lon, ST_AsText(geometry) AS geom_wkt FROM {TOWERS_TABLE} LIMIT 5").show(truncate=False)

### Step 4 — Validation
Confirm all three tables are populated and ready for downstream H3 and ST function analysis.

In [0]:
# ── Validate all ingested tables ─────────────────────────────────────────────
print("=" * 70)
print("INGESTION VALIDATION SUMMARY")
print("=" * 70)

for table_name, desc in [
    (FCC_TABLE, "FCC BDC 5G NR H3 Coverage"),
    (BUILDINGS_TABLE, "Microsoft Building Footprints"),
    (TOWERS_TABLE, "OpenCellID T-Mobile Towers"),
]:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {table_name}").collect()[0]["cnt"]
    cols_df = spark.sql(f"DESCRIBE {table_name}").collect()
    col_names = [row[0] for row in cols_df]
    print(f"\n✅ {desc}")
    print(f"   Table:   {table_name}")
    print(f"   Rows:    {count:,}")
    print(f"   Columns: {col_names}")

# Quick H3 function test on FCC table
print("\n" + "-" * 70)
print("H3 function test (FCC BDC table):")
spark.sql(f"""
    SELECT h3_res9_id,
           h3_h3tostring(h3_stringtoh3(h3_res9_id)) AS roundtrip,
           h3_resolution(h3_stringtoh3(h3_res9_id)) AS resolution
    FROM {FCC_TABLE}
    LIMIT 3
""").show(truncate=False)

# Quick ST function test on buildings table
print("ST function test (Building footprints table):")
spark.sql(f"""
    SELECT height,
           ST_Area(geometry) AS area_sq_deg,
           ST_AsText(ST_Centroid(geometry)) AS centroid_wkt
    FROM {BUILDINGS_TABLE}
    LIMIT 3
""").show(truncate=False)

# Quick ST + H3 function test on towers table
print("ST + H3 function test (Towers table):")
spark.sql(f"""
    SELECT radio, lat, lon,
           ST_AsText(geometry) AS point_wkt,
           h3_longlatash3string(lon, lat, 9) AS h3_index_res9
    FROM {TOWERS_TABLE}
    LIMIT 3
""").show(truncate=False)

print("\n" + "=" * 70)
print("✅ All tables ingested and ready for downstream H3 / ST analysis!")